In [2]:
import random
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from wfcrl.rewards import StepPercentage
from wfcrl import environments as envs

import torch
from pfns4bo import priors, encoders, utils, bar_distribution, train
from pfns4bo.priors.utils import Batch
from ConfigSpace import hyperparameters as CSH

import warnings
warnings.filterwarnings("ignore")

SEED = 13
N_sim = 1000 # 100_000

In [2]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
device = 'cuda'

# Get Simulation Data

In [5]:
def step_policy(i):
    '''Randmoly samples actions'''
    joint_action = {"yaw": np.zeros(env.num_turbines)}
    mask = np.random.random(env.num_turbines) < 0.3 # proba of a turbine to get controled
    joint_action["yaw"][mask] = np.random.uniform(-5, 5, size=mask.sum())
    return joint_action

def obs_to_row(obs, reward, power, step):
    '''Writes all episode results into a dict'''
    row = {"step": step, "reward": reward[0]}
    for key, val in obs.items():
        val = np.atleast_1d(val)
        if len(val) == 1:
            row[key] = val[0]
        else:
            for i, v in enumerate(val):
                row[f"{key}_{i}"] = v
    for i in range(len(power)):
        row[f"power_{i}"] = power[i]
    return row

In [6]:
env = envs.make("Turb3_Row1_Floris", #"Ablaincourt_Floris",#
                max_num_steps=150,
                controls={"yaw": (-5, 5)},
                continuous_control = True, # continuous action space
                log=True)

In [7]:
sims, max_reward, rows = 0, -np.inf, []
while len(rows) < N_sim:
    observation = env.reset(seed=SEED, options={"wind_speed": 8, "wind_direction": 270}) # to fix initial coniditions (Sc.1): options={"wind_speed": 8, "wind_direction": 270}
    r, i = 0, 0
    done = False
    while not done:
        joint_action = step_policy(i)
        observation, reward, termination, truncation, info = env.step(joint_action)
        rows.append(obs_to_row(observation, reward, info["power"], i))  
        r += reward
        i += 1
        done = termination or truncation
    if sims%500==0:
        print(f"Simulation #{sims}: total reward = {r}")
    sims += 1 
    max_reward = max(max_reward, r)
print(f"Maximum total reward: {max_reward}")
sim_summary = pd.DataFrame(rows)
sim_summary

Simulation #0: total reward = [228.7826061]
Maximum total reward: [230.46147907]


,step,reward,yaw_0,yaw_1,yaw_2,freewind_measurements_0,freewind_measurements_1,wind_speed_0,wind_speed_1,wind_speed_2,wind_direction_0,wind_direction_1,wind_direction_2,power_0,power_1,power_2
0,0,1.530322,0.000000,1.000000,0.129792,8.0,270.0,7.973633,4.896107,4.737564,270.264106,270.879848,270.951887,1.691327,0.362622,0.324714
1,1,1.529070,0.000000,0.000000,-0.052364,8.0,270.0,7.973633,4.896107,4.728997,270.264106,270.590438,270.704399,1.691327,0.362734,0.322661
2,2,1.529070,0.000000,0.000000,-0.052364,8.0,270.0,7.973633,4.896107,4.728997,270.264106,270.590438,270.704399,1.691327,0.362734,0.322661
3,3,1.529070,0.000000,0.000000,-0.052364,8.0,270.0,7.973633,4.896107,4.728997,270.264106,270.590438,270.704399,1.691327,0.362734,0.322661
4,4,1.529070,0.000000,0.000000,-0.052364,8.0,270.0,7.973633,4.896107,4.728997,270.264106,270.590438,270.704399,1.691327,0.362734,0.322661
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1038,144,1.530011,-1.006463,-3.637153,-3.192710,8.0,270.0,7.973633,4.897796,4.747930,270.118218,269.377953,268.946152,1.690865,0.361656,0.326094
1039,145,1.530706,-0.006463,-4.637153,-3.192710,8.0,270.0,7.973633,4.896101,4.755773,270.263170,269.255114,268.881485,1.691327,0.360323,0.327973
1040,146,1.530913,-0.006463,-5.000000,-4.192710,8.0,270.0,7.973633,4.896101,4.762470,270.263170,269.152394,268.523510,1.691327,0.359931,0.328772
1041,147,1.530913,-0.006463,-5.000000,-4.192710,8.0,270.0,7.973633,4.896101,4.762470,270.263170,269.152394,268.523510,1.691327,0.359931,0.328772


# Train PFN4BO

In [22]:
def get_batch(batch_size, seq_len, num_features, device, hyperparameters, **kwargs):
    X_out = torch.zeros(seq_len, batch_size, num_features)
    Y_out = torch.zeros(seq_len, batch_size)
    
    for b in range(batch_size):
        idx = torch.randperm(len(X_all))[:seq_len]
        X_out[:, b, :] = X_all[idx]
        Y_out[:, b] = y_all_norm[idx]
    
    return Batch(x=X_out.to(device), y=Y_out.to(device), target_y=Y_out.to(device))

In [21]:
X = sim_summary[[f"yaw_{i}" for i in range(env.num_turbines)]+['freewind_measurements_0', 'freewind_measurements_1']]
y = sim_summary[[f'power_{i}' for i in range(env.num_turbines)]]

X_all = torch.tensor(X.values, dtype=torch.float32)  # (N, num_features)
y_all = torch.tensor(y.sum(axis=1).values, dtype=torch.float32)  # (N,) total power
y_mean = y_all.mean()
y_std = y_all.std()
y_all_norm = (y_all - y_mean) / y_std

num_features = env.num_turbines + 2

In [28]:
config = {
    'priordataloader_class': priors.get_batch_to_dataloader(get_batch),
    'encoder_generator': encoders.get_normalized_uniform_encoder(
        encoders.get_variable_num_features_encoder(encoders.Linear)
    ),
    'y_encoder_generator': encoders.Linear,
    'emsize': 512,
    'nhead': 4,
    'nhid': 1024,
    'nlayers': 6,               # 12 in the paper; halved for faster iteration
    'epochs': 1, #50
    'lr': 0.0001,
    'bptt': 150,                 # max context length
    'batch_size': 128,
    'steps_per_epoch': 512,
    'warmup_epochs': 5,
    'scheduler': utils.get_cosine_schedule_with_warmup,
    'aggregate_k_gradients': 2,
    'weight_decay': 0.0,
    'train_mixed_precision': False,
    'efficient_eval_masking': True,
    'single_eval_pos_gen': utils.get_uniform_single_eval_pos_sampler(50, min_len=1),
    'extra_prior_kwargs_dict': {
        'num_features': num_features,
        'hyperparameters': {}
    },
}

In [29]:
# ---- Criterion (bucket limits derived from your data distribution) ----
def get_ys(config, device):
    b = config['priordataloader_class'].get_batch_method(
        128, 1000, num_features, epoch=0, device=device,
        hyperparameters={**config['extra_prior_kwargs_dict']['hyperparameters'],
                         'num_hyperparameter_samples_per_batch': -1}
    )
    return b.target_y.flatten()

def add_criterion(config, device):
    return {**config, 'criterion': bar_distribution.FullSupportBarDistribution(
        bar_distribution.get_bucket_limits(100, ys=get_ys(config, device).cpu())
    )}

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
model = train.train(**add_criterion(config, device))

Using 128000 y evals to estimate 100 buckets. Cut off the last 0 ys.
Using cuda:0 device
init dist
Not using distributed
DataLoader.__dict__ {'num_steps': 512, 'get_batch_kwargs': {'batch_size': 128, 'eval_pos_seq_len_sampler': <function train.<locals>.eval_pos_seq_len_sampler at 0x7f2278656cb0>, 'seq_len_maximum': 150, 'device': 'cuda:0', 'num_features': 5, 'hyperparameters': {}}, 'num_features': 5, 'epoch_count': 0}
Style definition of first 3 examples: None
Initialized decoder for standard with (None, 100)  and nout 100
Using a Transformer with 13.25 M parameters
-----------------------------------------------------------------------------------------
| end of epoch   1 | time: 55.59s | mean loss  1.01 | pos losses   nan, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.00, 1.01, 1.01, 1.01, 1.00, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.02, 1.00, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1.01, 1

In [40]:
X_all = torch.tensor(X.values, dtype=torch.float32)
y_raw = torch.tensor(y.sum(axis=1).values, dtype=torch.float32)
y_norm = ((y_raw - y_mean) / y_std).unsqueeze(1)  # (N, 1)
X_ctx = X_all.unsqueeze(1).to(device)              # (N, 1, n_features)
y_ctx = y_ctx = ((y_raw - y_mean) / y_std).unsqueeze(1).to(device)  # (N, 1)

obs = env.reset(seed=27)
freewind = torch.tensor(obs['freewind_measurements'], dtype=torch.float32).to(device)

yaws = torch.zeros(3, dtype=torch.float32, device=device, requires_grad=True)
opt = torch.optim.Adam([yaws], lr=1.0)

In [41]:
x_query = torch.cat([yaws, freewind]).reshape(1, 1, -1)
x_full = torch.cat([X_ctx, x_query], dim=0)
y_full = torch.cat([y_ctx, torch.zeros(1, 1, device=device)], dim=0)

In [57]:
type(p)

NoneType

In [58]:
x_full.shape, y_full.shape

(torch.Size([1044, 1, 5]), torch.Size([1044, 1]))

In [56]:
p = model[2](x_full, y_full, only_return_standard_out=False)
p

In [26]:
def run_pfn(env, pfn_model, seed, device, n_initial=10):
    X_context, y_context = [], []

    # --- Random exploration ---
    obs = env.reset(seed=seed, options=OPTIONS)
    freewind = obs['freewind_measurements']

    for _ in range(n_initial):
        yaws = np.random.uniform(-40, 40, nt)
        r = ramp_and_eval(env, yaws, seed)
        X_context.append(np.concatenate([yaws, freewind]))
        y_context.append(r)

    # --- Optimize yaws once through PFN ---
    X_t = torch.tensor(np.array(X_context), dtype=torch.float32).unsqueeze(1).to(device)
    y_t = torch.tensor((np.array(y_context) - y_mean) / y_std, dtype=torch.float32).unsqueeze(1).to(device)

    obs = env.reset(seed=seed, options=OPTIONS)
    freewind_t = torch.tensor(obs['freewind_measurements'], dtype=torch.float32).to(device)

    yaws = torch.zeros(nt, dtype=torch.float32, device=device, requires_grad=True)
    opt = torch.optim.Adam([yaws], lr=1.0)
    pfn_model.eval()

    for _ in range(100):
        opt.zero_grad()
        x_query = torch.cat([yaws, freewind_t]).reshape(1, 1, -1)
        x_full = torch.cat([X_t, x_query], dim=0)
        y_full = torch.cat([y_t, torch.zeros(1, 1, device=device)], dim=0)
        logits = pfn_model(x_full, y_full, single_eval_pos=len(X_context))
        (-pfn_model.criterion.mean(logits)).backward()
        opt.step()
        yaws.data.clamp_(-40, 40)

    return ramp_and_eval(env, yaws.detach().cpu().numpy(), seed)


def ramp_and_eval(env, target_yaws, seed):
    obs = env.reset(seed=seed, options=OPTIONS)
    r, done = 0, False
    while not done:
        delta = np.clip(target_yaws - obs['yaw'], -5, 5)
        obs, reward, term, trunc, _ = env.step({'yaw': delta})
        r += reward
        done = term or trunc
    return r[0]

Using 128000 y evals to estimate 1000 buckets. Cut off the last 0 ys.
tensor([-1.2306, -1.2173, -1.2104, -1.2104, -1.1986]) tensor([2.8140, 2.8140, 2.8140, 2.8140, 2.8140])
tensor(0.)


In [ ]:
optimize_with_pfn(env, model, X_context, y_context_norm, seed, device)

In [6]:
from sklearn.datasets import fetch_openml
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

In [16]:
observation = env.reset(seed=SEED,options={"wind_speed": 8, "wind_direction": 270})
r, i, power = 0, 0, 0
done = False
while not done:

    freewind = torch.tensor(observation['freewind_measurements'], dtype=torch.double)
    current_yaw = observation['yaw']
    
    if i==0:
            yaws = torch.nn.Parameter(torch.zeros(1, 1, num_features, device=device))
    optimizer = torch.optim.Adam([yaws], lr=0.01)

    for _ in range(200):
        x_full = torch.cat([x_ctx, yaws], dim=0)
        logits = model(x_full, y_full, single_eval_pos=len(X_context))
        loss = -model.criterion.mean(logits)  # maximize predicted power
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        yaws.data.clamp_(-40, 40)  # enforce yaw bounds

    #if i == 0:
    joint_action = {'yaw': result['x'] - current_yaw}
    #else:
        #joint_action = {'yaw': np.zeros(env.num_turbines)}
        
    observation, reward, termination, truncation, info = env.step(joint_action)
    r += reward
    power += info["power"]
    i += 1
    done = termination or truncation
print(f"Total reward = {r}\tTotal power= {sum(power)}")

Total reward = [351.16638054]	Total power= 1271.6118774242086


In [18]:
joint_action

{'yaw': array([ 4.52639592e-11,  8.67450209e-13, -2.93722381e-12,  7.28547225e-12,
        -9.60980601e-12, -8.19204036e-12,  7.71880021e-12])}

In [15]:
obs, reward, _, _, info = env.step({'yaw': np.zeros(env.num_turbines)})
print(f"step reward: {reward}")
print(f"step power: {info.get('power', 'N/A')}")
print(f"step loads: {info.get('load', 'N/A')}")

step reward: [2.35682365]
step power: [1.69132665 0.39630197 1.68870792 1.3113222  1.30026544 1.20969173
 0.93669921]
step loads: [[0.06020304 0.2854754  0.09781062 0.0520248 ]
 [0.11336189 1.10848606 0.13966051 0.06096605]
 [0.06047356 0.28533935 0.10218463 0.04875879]
 [0.11098817 0.72576413 0.10929816 0.03985623]
 [0.11330846 0.70562934 0.10991932 0.03930753]
 [0.11342138 0.82187501 0.11171011 0.03783295]
 [0.11431011 1.11026893 0.11963405 0.03487882]]
